In [1]:
from pathlib import Path
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from IPython.display import Image, display
from dotenv import load_dotenv
from google import genai
from google.genai import types
from dotenv import load_dotenv
import base64
import os
load_dotenv()



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [15]:
if "__file__" in globals():
    base_path = Path(__file__).parent
else:
    base_path = Path.cwd()

data_folder = base_path.parent / "Papers"
files=list(data_folder.glob("*.pdf"))
print(files[2])




/Users/shagungothwal/Downloads/Python/RAG_project/Papers/AI_Agents.pdf


In [7]:

elements = partition_pdf(
    filename=files[4],                 
    strategy="hi_res",                                    
    extract_images_in_pdf=True,  
    extract_image_block_types=["Image", "Table"],          
    extract_image_block_to_payload=True, 
    

)
elements = [e for e in elements if e.category not in ("Header", "Footer")]




No languages specified, defaulting to English.


KeyboardInterrupt: 

In [10]:

chunks = chunk_by_title(elements,
                        include_orig_elements=True,
                        max_characters=2500,
                        combine_text_under_n_chars=500,
                        new_after_n_chars=2000,
                        overlap=300)



In [89]:
for chunk in chunks:
    print(chunk.id)


0bfc1419-159e-4deb-9544-75b5d6b8a032
7aa950f1-01ad-424c-ab8c-0988c82fda07
753963e4-79e8-44ff-993e-1afa5ce27c7b
5611f2a9-55a7-4232-a212-5599184420d0
214e7c7e-15bb-4d83-9577-6655c1e3e529
ceafff98-18f7-4599-9c2c-449b94b2a505
8003c1cb-5e14-4cbd-a712-5cbf2d401536
d40961f4-e041-4beb-a0d7-67e7e5dba2cc
831fe62f-51f4-41c7-870c-b45ccf01bca9
ede65ea0-5f12-416e-a727-5ad0c3aa1bfa
206a8d6b-40fd-4834-9049-07f9d2f5f957
2863f5ad-ef0f-41e3-8f48-2d44616c2a16
9bc7cdc2-3448-41ec-9e46-73f1eb9bbf72
51366482-5d76-4775-a16c-32bf38e130b3
9e27d4ad-0465-4d0c-aa12-75defccf7bee
d4ad68fd-82e0-4cd4-98a6-f4f5b91c2f3d
e3c1be9d-f248-4e5a-bde8-26e2a47a5867
19782179-20ca-4bb5-a6dc-994c679563da
2e20cb6a-9c05-401c-af7b-d0598bdea81f
9691e3dd-a823-4dff-af04-cdf80ab26702
e8488814-4dd8-4459-8213-7aefcf0a630e
ffb3a0e3-3867-40a3-a33f-bf650684c183
7dec6da1-34b8-4ba9-b9b8-07de2c4e62c8
bfaa6588-3928-4dd1-bc8b-91189e36c7b5


In [66]:
page=set()
for ele in chunks[10].metadata.orig_elements:
    print(ele.category)
    page.add(ele.metadata.page_number)
page    
  

Title
Title
NarrativeText
NarrativeText


{8, 9}

In [ ]:
all_chunks=[]
def extractor(chunk,i,c_length):
    text=chunk.text
    table=[]
    img=[]
   
    print(f"processing chunk ⏳:{i}/{c_length}")
    for ele in chunk.metadata.orig_elements:
           
        ele_type=ele.category
        try:
            if ele_type=='Table':
                print(f"!!!found Table to Process in chunk {i}!!!\n\n")
                
                img64=ele.metadata.image_base64
                imgurl=ele.metadata.image_url
                if imgurl or img64: #some tables exist as img
                    image_url=getattr(ele.metadata,'image_base64','image_url')
                    img.append(image_url) 
                   
                else:    
                    table_data=getattr(ele.metadata,'text_as_html','text')
                    table.append(table_data)

            elif ele_type=='Image':
                print(f"!!!found Image 🏙️ to Process in chunk {i}!!!\n\n")
                image_url=getattr(ele.metadata,'image_base64','image_url')    
                img.append(image_url)
        
        except Exception as e: 
            print("error",e)
     

    return({"text":text,
                "table":table,
                "img":img})        
def summarise(chunk,i,c_len):
    print(f"----sending chunk {i}/{c_len} with tables and images to AI 🤖 for summarising----")
    
    text=chunk['text']
    images=chunk['img']
    tables=chunk['table']
    contents=[]
    try:
       
        SYSTEM_PROMPT=f"""Here You are provided with some data, it includes image and/or table along with it you are being provided with some text data your task is:
        a)Analyze the image and/or table alongside with the text provided
        b)Come up with a comprhensive summary after analyzing everything 
        c)NEVER go beyond the scope of text, image and/or table and DO NOT add uncessary details
        \nTEXT:{text}\n\n
        """
        if(tables):
            SYSTEM_PROMPT+=f"""TABLES\n\n"""
            for i,table in enumerate(tables):
                SYSTEM_PROMPT+=f'''Table{i+1}:{table} \n\n\n'''
        if(images):  
            SYSTEM_PROMPT+=f"""IMAGES\n\n"""
            for image in images:
                contents.append(
                    types.Part.from_bytes(
                        data=base64.b64decode(image),
                        mime_type="image/jpeg"
                    )
                )
            SYSTEM_PROMPT+=f"""{contents}"""   
            
        client = genai.Client()
        response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=SYSTEM_PROMPT
        )

        print(response.text[:100])
        summary=response.text
        chunk['text'].clear()
        print(chunk['text'][:100])
        chunk['text']=summary
        return(chunk)
    except Exception as e:
        print("error",e)


def loader(chunks):
    
    c_len=len(chunks)
    for i,chunk in enumerate(chunks):
        f_chunk=extractor(chunk,i+1,c_len)
        if(f_chunk['table']or f_chunk['img']):
            AI_chunk=summarise(f_chunk,i+1,c_len)
            all_chunks.append(AI_chunk)

            
        else:
            all_chunks.append(f_chunk)
    

processeing chunk ⏳:1/21
processeing chunk ⏳:2/21
!!!found Image 🏙️ to Process in chunk 2!!!


!!!found Image 🏙️ to Process in chunk 2!!!


----sending chunk 2/21 with tables and images to AI 🤖 for summarising----
processeing chunk ⏳:3/21
processeing chunk ⏳:4/21
processeing chunk ⏳:5/21
processeing chunk ⏳:6/21
processeing chunk ⏳:7/21
!!!found Table to Process in chunk 7!!!


----sending chunk 7/21 with tables and images to AI 🤖 for summarising----
processeing chunk ⏳:8/21
processeing chunk ⏳:9/21
!!!found Image 🏙️ to Process in chunk 9!!!


----sending chunk 9/21 with tables and images to AI 🤖 for summarising----
processeing chunk ⏳:10/21
!!!found Image 🏙️ to Process in chunk 10!!!


----sending chunk 10/21 with tables and images to AI 🤖 for summarising----
processeing chunk ⏳:11/21
!!!found Image 🏙️ to Process in chunk 11!!!


----sending chunk 11/21 with tables and images to AI 🤖 for summarising----
processeing chunk ⏳:12/21
!!!found Image 🏙️ to Process in chunk 12!!!


----sendin

In [ ]:
elements = partition_pdf(
    filename=files[2],                 
    strategy="hi_res",                                    
    extract_images_in_pdf=True,  
    extract_image_block_types=["Image", "Table"],          
    extract_image_block_to_payload=True, 
    

)
elements = [e for e in elements if e.category not in ("Header", "Footer")]
print(elements)


No languages specified, defaulting to English.


In [41]:
chunks = chunk_by_title(elements,
                        include_orig_elements=True,
                        max_characters=1200,
                        combine_text_under_n_chars=500,
                        new_after_n_chars=1000,
                        overlap=150)

In [ ]:
chunks[20].to_dict()
chunks[28].metadata.orig_elements[3].

{'type': 'NarrativeText',
 'element_id': 'd80955d3aa0cf98bcbfd65dd333b78cb',
 'text': 'missing one of the terms. We leave the computation for Documents 3 and 4 as an exercise for the reader.',
 'metadata': {'detection_class_prob': 0.9107551574707031,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(675.2257690429688),
     np.float64(1986.110591111111)),
    (np.float64(675.2257690429688), np.float64(2092.6544799999997)),
    (np.float64(2316.820556640625), np.float64(2092.6544799999997)),
    (np.float64(2316.820556640625), np.float64(1986.110591111111))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-11T23:48:21',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 8,
  'file_directory': '/Users/shagungothwal/Downloads/Python/RAG_project/Papers',
  'filename': 'RAG.pdf'}}

In [ ]:
##chunks[27].metadata.orig_elements[1].to_dict() -237c0ab46d1ea6a053ac32b22937085c

In [11]:
from langchain_core.documents import Document

In [16]:
docu_list=[]
for chunk in chunks:
    
    extra_data={'id':chunk.id,'page':chunk.metadata.page_number}
    doc=Document(page_content=chunk.text,
             metadata=extra_data
             )
    docu_list.append(doc)

In [20]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [21]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5", model_kwargs={'device': 'mps'})

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8646.17it/s]


In [23]:
vector_store=Chroma.from_documents(documents=docu_list,embedding=embeddings,persist_directory='./')

In [65]:
res=vector_store.similarity_search('what is cot',k=20)